# Segmenting and Clustering Neighborhoods in Toronto

In [4]:
import json
import requests

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Geocoding
from geopy.geocoders import Nominatim

# JSON normalization
from pandas import json_normalize

# Plotting
import matplotlib.cm as cm
import matplotlib.colors as colors

# Machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Mapping
import folium

print("Libraries imported.")

Libraries imported.


### Get dataset 1: Postal Code, Borough, and Neighborhood

In [12]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_postal_codes_of_Canada:_M"

headers = {
    "User-Agent": "Mozilla/5.0"
}

req = requests.get(url, headers=headers)
req.raise_for_status()

soup = BeautifulSoup(req.text, "html.parser")

table = soup.find("table")

tables = pd.read_html(StringIO(str(table)))

neighborhood = tables[0]

neighborhood.head()

,0,1,2,3,4,5,6,7,8
0,M1A Not assigned,M2A Not assigned,M3A North York (Parkwoods),M4A North York (Victoria Village),M5A Downtown Toronto (Regent Park / Harbourfront),M6A North York (Lawrence Manor / Lawrence Heig...,M7A Queen's ParkOntario Provincial Government,M8A Not assigned,M9A Etobicoke (Islington Avenue)
1,M1B Scarborough (Malvern / Rouge),M2B Not assigned,M3B North York (Don Mills) North,M4B East York (Parkview Hill / Woodbine Gardens),"M5B Downtown Toronto (Garden District, Ryerson)",M6B North York (Glencairn),M7B Not assigned,M8B Not assigned,M9B Etobicoke (West Deane Park / Princess Gard...
2,M1C Scarborough (Rouge Hill / Port Union / Hig...,M2C Not assigned,M3C North York (Don Mills) South (Flemingdon P...,M4C East York (Woodbine Heights),M5C Downtown Toronto (St. James Town),M6C York (Humewood-Cedarvale),M7C Not assigned,M8C Not assigned,M9C Etobicoke (Eringate / Bloordale Gardens / ...
3,M1E Scarborough (Guildwood / Morningside / Wes...,M2E Not assigned,M3E Not assigned,M4E East Toronto (The Beaches),M5E Downtown Toronto (Berczy Park),M6E York (Caledonia-Fairbanks),M7E Not assigned,M8E Not assigned,M9E Not assigned
4,M1G Scarborough (Woburn),M2G Not assigned,M3G Not assigned,M4G East York (Leaside),M5G Downtown Toronto (Central Bay Street),M6G Downtown Toronto (Christie),M7G Not assigned,M8G Not assigned,M9G Not assigned


In [21]:
wide_df = neighborhood.copy()

# 1. Convert wide table to one column
data = (
    wide_df
    .stack()
    .reset_index(drop=True)
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .to_frame(name="raw")
)

# 2. Remove Not assigned rows
data = data[
    ~data["raw"].str.contains("Not assigned", case=False, na=False)
].copy()

# 3. Extract postal code
data["Postal code"] = data["raw"].str.extract(r"^([A-Z]\d[A-Z])")

# 4. Remove postal code from raw text
data["details"] = data["raw"].str.replace(
    r"^[A-Z]\d[A-Z]\s*",
    "",
    regex=True )

# 5. Extract Borough and Neighborhood
parsed = data["details"].str.extract(
    r"^(?P<Borough>.*?)\s*\((?P<Neighborhood>.*?)\)\s*$" )

data["Borough"] = parsed["Borough"].fillna(data["details"])
data["Neighborhood"] = parsed["Neighborhood"].fillna(data["details"])

# 6. Clean neighborhood separators
data["Neighborhood"] = data["Neighborhood"].str.replace(" / ", ", ", regex=False)

# 7. Special case: M7A
mask = data["Postal code"].eq("M7A") & data["details"].str.contains(
    "Ontario Provincial Government",
    na=False )

data.loc[mask, "Borough"] = "Queen's Park"
data.loc[mask, "Neighborhood"] = "Ontario Provincial Government"

In [22]:
# 8. Final clean table
postal_df = data[["Postal code", "Borough", "Neighborhood"]].reset_index(drop=True)

postal_df.head()

,Postal code,Borough,Neighborhood
0,M3A,North York,Parkwoods
1,M4A,North York,Victoria Village
2,M5A,Downtown Toronto,"Regent Park, Harbourfront"
3,M6A,North York,"Lawrence Manor, Lawrence Heights"
4,M7A,Queen's Park,Ontario Provincial Government


In [23]:
data.info()

<class 'pandas.DataFrame'>
Index: 103 entries, 2 to 178
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   raw           103 non-null    str  
 1   Postal code   103 non-null    str  
 2   details       103 non-null    str  
 3   Borough       103 non-null    str  
 4   Neighborhood  103 non-null    str  
dtypes: str(5)
memory usage: 19.4 KB


In [24]:
postal_df.shape

(103, 3)

### Get dataset 2: The latitude and the longitude coordinates of each neighborhood.

In [25]:
df = pd.read_csv('http://cocl.us/Geospatial_data')

In [26]:
df.head()

,Postal Code,Latitude,Longitude
0,M1B,43.806686,-79.194353
1,M1C,43.784535,-79.160497
2,M1E,43.763573,-79.188711
3,M1G,43.770992,-79.216917
4,M1H,43.773136,-79.239476


In [27]:
merge = pd.merge(left=data, right=df, how='left', left_on='Postal code', right_on='Postal Code')
merge.head()

,raw,Postal code,details,Borough,Neighborhood,Postal Code,Latitude,Longitude
0,M3A North York (Parkwoods),M3A,North York (Parkwoods),North York,Parkwoods,M3A,43.753259,-79.329656
1,M4A North York (Victoria Village),M4A,North York (Victoria Village),North York,Victoria Village,M4A,43.725882,-79.315572
2,M5A Downtown Toronto (Regent Park / Harbourfront),M5A,Downtown Toronto (Regent Park / Harbourfront),Downtown Toronto,"Regent Park, Harbourfront",M5A,43.654260,-79.360636
3,M6A North York (Lawrence Manor / Lawrence Heig...,M6A,North York (Lawrence Manor / Lawrence Heights),North York,"Lawrence Manor, Lawrence Heights",M6A,43.718518,-79.464763
4,M7A Queen's ParkOntario Provincial Government,M7A,Queen's ParkOntario Provincial Government,Queen's Park,Ontario Provincial Government,M7A,43.662301,-79.389494


In [28]:
geo = merge.drop('Postal Code', axis = 1)
geo.head()

,raw,Postal code,details,Borough,Neighborhood,Latitude,Longitude
0,M3A North York (Parkwoods),M3A,North York (Parkwoods),North York,Parkwoods,43.753259,-79.329656
1,M4A North York (Victoria Village),M4A,North York (Victoria Village),North York,Victoria Village,43.725882,-79.315572
2,M5A Downtown Toronto (Regent Park / Harbourfront),M5A,Downtown Toronto (Regent Park / Harbourfront),Downtown Toronto,"Regent Park, Harbourfront",43.654260,-79.360636
3,M6A North York (Lawrence Manor / Lawrence Heig...,M6A,North York (Lawrence Manor / Lawrence Heights),North York,"Lawrence Manor, Lawrence Heights",43.718518,-79.464763
4,M7A Queen's ParkOntario Provincial Government,M7A,Queen's ParkOntario Provincial Government,Queen's Park,Ontario Provincial Government,43.662301,-79.389494


### Explore and cluster the neighborhoods in Toronto

In [30]:
address = "Toronto, Canada"

geolocator = Nominatim(user_agent="toronto_clustering_project")
location = geolocator.geocode(address)

if location is not None:
    latitude = location.latitude
    longitude = location.longitude
    
    print(f"The geographical coordinates of Toronto are {latitude}, {longitude}.")
else:
    print("Location not found. Try a more specific address.")

The geographical coordinates of Toronto are 43.6534817, -79.3839347.


### Create a map of Toronto with neighborhoods superimposed on top

In [32]:
geo[["Latitude", "Longitude"]].isna().sum()

Latitude     0
Longitude    0
dtype: int64

In [31]:
# Create map of Toronto using latitude and longitude
map_toronto = folium.Map(
    location=[latitude, longitude],
    zoom_start=10
)

# Add neighborhood markers to the map
for lat, lng, borough, neighborhood in zip(
    geo["Latitude"],
    geo["Longitude"],
    geo["Borough"],
    geo["Neighborhood"]
):
    label = f"{neighborhood}, {borough}"
    label = folium.Popup(label, parse_html=True)
    
    folium.CircleMarker(
        location=[lat, lng],
        radius=5,
        popup=label,
        color="blue",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7
    ).add_to(map_toronto)

map_toronto

### Save map as html

In [33]:
from branca.element import Element

title_html = """
<div style="
    position: fixed;
    top: 10px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;
    background-color: white;
    padding: 10px 18px;
    border-radius: 6px;
    font-size: 22px;
    font-weight: bold;
    box-shadow: 0px 0px 6px rgba(0,0,0,0.3);
">
    Toronto Neighborhood Map
</div>
"""

map_toronto.get_root().html.add_child(Element(title_html))

map_toronto

In [35]:
from pathlib import Path

output_dir = Path("toronto_clustering")
output_dir.mkdir(parents=True, exist_ok=True)

map_toronto.save(output_dir / "toronto_neighborhood_map.html")